## Setup and Imports
Importi:
transformers per tokenizer/modello/Trainer
torch per training/inferenza
utility (json, os, numpy, pandas, tqdm)

In [91]:
import json
import os
import numpy as np
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
from torch.utils.data import Dataset
import pandas as pd
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Setup complete
PyTorch version: 2.10.0+cpu
CUDA available: False


In [92]:
# Define entity labels
ENTITY_LABELS = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "DDF",
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique"
]

# Create BIO tags for each entity label
label_list = ['O']  # Outside
for entity_label in ENTITY_LABELS:
    label_list.append(f'B-{entity_label}')  # Beginning
    label_list.append(f'I-{entity_label}')  # Inside

label2id = {k: v for v, k in enumerate(label_list)}
id2label = {v: k for v, k in enumerate(label_list)}

print(f"Total labels: {len(label_list)}")
print(f"\nFirst 10 labels: {label_list[:10]}")

# Model configuration
model_name = "dmis-lab/biobert-v1.1"  # BioBERT for biomedical text
output_model_dir = "models/bert_biomedbert_ner_aligned"

print(f"\nModel: {model_name}")
print(f"Output directory: {output_model_dir}")

Total labels: 27

First 10 labels: ['O', 'B-anatomical location', 'I-anatomical location', 'B-animal', 'I-animal', 'B-bacteria', 'I-bacteria', 'B-biomedical technique', 'I-biomedical technique', 'B-chemical']

Model: dmis-lab/biobert-v1.1
Output directory: models/bert_biomedbert_ner_aligned


## Data Loading Functions

In [93]:
def load_ner_data(file_paths):
    """
    Load NER data from multiple JSON files.
    Each file contains documents with entities.
    """
    all_data = {}
    
    for file_path in file_paths:
        if os.path.exists(file_path):
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            all_data.update(data)
            print(f"Loaded {len(data)} documents from {os.path.basename(file_path)}")
        else:
            print(f"Warning: {file_path} not found")
    
    return all_data


def prepare_documents_for_ner(data):
    """
    Convert raw data into structured format for NER.
    Each document has title and abstract as separate text segments.
    """
    documents = []
    
    for pmid, article in data.items():
        # Process title
        title_text = article['metadata']['title']
        title_entities = [e for e in article['entities'] if e['location'] == 'title']
        
        documents.append({
            'pmid': pmid,
            'location': 'title',
            'text': title_text,
            'entities': title_entities
        })
        
        # Process abstract
        abstract_text = article['metadata']['abstract']
        abstract_entities = [e for e in article['entities'] if e['location'] == 'abstract']
        
        documents.append({
            'pmid': pmid,
            'location': 'abstract',
            'text': abstract_text,
            'entities': abstract_entities
        })
    
    return documents


print("✓ Data loading functions defined")

✓ Data loading functions defined


## Load Training and Dev Data

In [94]:
import json
from pathlib import Path
from collections import Counter, defaultdict
PROJECT_ROOT = Path.cwd().parents[1]
DATA_ROOT = PROJECT_ROOT / "data" / "GutBrainIE_Full_Collection_2025"
ANNOTATIONS_DIR = DATA_ROOT / "Annotations"

train_files = [
    ANNOTATIONS_DIR / "Train" / "gold_quality" / "json_format" / "train_gold.json",
    ANNOTATIONS_DIR / "Train" / "platinum_quality" / "json_format" / "train_platinum.json",
    ANNOTATIONS_DIR / "Train" / "silver_quality" / "json_format" / "train_silver.json",
]

train_data = load_ner_data(train_files)
train_documents = prepare_documents_for_ner(train_data)

print(f"\nTotal training documents: {len(train_documents)}")
print(f"Total training text segments: {len(train_documents)}")

Loaded 208 documents from train_gold.json
Loaded 111 documents from train_platinum.json
Loaded 499 documents from train_silver.json

Total training documents: 1636
Total training text segments: 1636


In [95]:
# Load dev data
dev_data_path = (
    ANNOTATIONS_DIR
    / "Dev"
    / "json_format"
    / "dev.json"
)
dev_data_path = (
    ANNOTATIONS_DIR
    / "Dev"
    / "json_format"
    / "dev.json"
)

with dev_data_path.open(encoding="utf-8") as f:
    dev_data = json.load(f)

dev_documents = prepare_documents_for_ner(dev_data)

print(f"Total dev documents: {len(dev_documents)}")

Total dev documents: 80


In [96]:
# Show example document
example_doc = train_documents[10]
print(f"Example document:")
print(f"  PMID: {example_doc['pmid']}")
print(f"  Location: {example_doc['location']}")
print(f"  Text: {example_doc['text'][:200]}...")
print(f"  Number of entities: {len(example_doc['entities'])}")
print(f"\nFirst 3 entities:")
for entity in example_doc['entities'][:3]:
    print(f"    - '{entity['text_span']}' [{entity['label']}] @ {entity['start_idx']}-{entity['end_idx']}")

Example document:
  PMID: 37127945
  Location: title
  Text: A systematic review on gut-brain axis aberrations in bipolar disorder and methods of balancing the gut microbiota....
  Number of entities: 2

First 3 entities:
    - 'bipolar disorder' [DDF] @ 53-68
    - 'gut microbiota' [microbiome] @ 99-112


## Initialize BERT Model and Tokenizer

In [97]:
# Initialize tokenizer and model
print("Initializing BERT tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

print(f"✓ Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"✓ Model loaded with {model.num_labels} labels")
print(f"  Vocabulary size: {tokenizer.vocab_size}")

# Test tokenization
sample_text = "The gut microbiome plays a role in Parkinson's disease."
tokens = tokenizer.tokenize(sample_text)
print(f"\nSample tokenization: {tokens}")

Initializing BERT tokenizer and model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 477.53it/s, Materializing param=bert.encoder.layer.11.output.dense.weight]              
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Tokenizer loaded: BertTokenizer
✓ Model loaded with 27 labels
  Vocabulary size: 28996

Sample tokenization: ['The', 'gut', 'micro', '##bio', '##me', 'plays', 'a', 'role', 'in', 'Parkinson', "'", 's', 'disease', '.']


## BIO Tag Generation for Training Data

align_labels_with_tokens(text, entities, tokenizer, label2id)

1. Tokenizza con:
encoding = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True, truncation=True, max_length=512)

offset_mapping ti dà per ogni token la coppia (start_char, end_char) nel testo originale.
2. Inizializza tutti i token a O.
3. Ordina le entità per start e poi per lunghezza decrescente:
sorted_entities = sorted(entities, key=lambda e: (e['start_idx'], -(e['end_idx'] - e['start_idx'])))
✅ così in caso di overlap provi a mettere prima le più lunghe.

4. Per ogni entità cerchi quali token “overlappano” lo span:
if token_start < entity_end and token_end > entity_start:
✅ overlap robusto.

5. Applichi BIO sui token trovati MA con vincolo “non etichettare due volte” usando labeled_positions.

In [98]:
def align_labels_with_tokens(text, entities, tokenizer, label2id, max_length=512):
    """
    Create BIO tags for tokenized text based on character-level entity annotations.

    Key fixes vs baseline:
    - Handles inclusive end_idx in dataset by converting to exclusive end for overlap checks.
    - Ignores special tokens and (optionally) can ignore subword-only labeling errors.
    - Deterministic overlap policy: longer spans first; do not overwrite already-labeled tokens.
    - Casts offsets to int for safety.
    """
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=max_length,
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    offset_mapping = encoding["offset_mapping"]  # list[(start,end)] end is exclusive

    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    # Initialize all labels as 'O'
    labels = ["O"] * len(input_ids)

    # Sort entities: earlier start first, then longer first (so we keep more specific spans)
    sorted_entities = sorted(
        entities,
        key=lambda e: (int(e["start_idx"]), -(int(e["end_idx"]) - int(e["start_idx"]))),
    )

    labeled_positions = set()

    for ent in sorted_entities:
        ent_start = int(ent["start_idx"])
        ent_end_excl = int(ent["end_idx"]) + 1  # ✅ dataset end_idx is inclusive → convert to exclusive
        ent_label = str(ent["label"])

        ent_token_start = None
        ent_token_end = None

        for idx, (tok_start, tok_end) in enumerate(offset_mapping):
            tok_start = int(tok_start)
            tok_end = int(tok_end)

            # Special tokens have (0,0) offsets in HF tokenizers
            if tok_start == 0 and tok_end == 0:
                continue

            # Robust overlap check (character spans)
            if tok_start < ent_end_excl and tok_end > ent_start:
                if ent_token_start is None:
                    ent_token_start = idx
                ent_token_end = idx

        # Apply BIO tags if we found any overlapping tokens
        if ent_token_start is not None and ent_token_end is not None:
            for i in range(ent_token_start, ent_token_end + 1):
                if i in labeled_positions:
                    continue

                if i == ent_token_start:
                    tag = f"B-{ent_label}"
                else:
                    tag = f"I-{ent_label}"

                # Fallback safety: if tag not in label2id, keep O
                if tag in label2id:
                    labels[i] = tag
                    labeled_positions.add(i)

    label_ids = [label2id.get(tag, label2id["O"]) for tag in labels]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": label_ids,
        "tokens": tokens,
    }


## Process Training and Dev Data with BIO Tags

In [99]:
# Process training data
print("Processing training data...")
processed_train = []

for i, doc in enumerate(tqdm(train_documents, desc="Processing train")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_train.append(processed)

print(f"✓ Training data processed: {len(processed_train)} segments")

Processing training data...


Processing train: 100%|██████████| 1636/1636 [00:03<00:00, 449.76it/s]

✓ Training data processed: 1636 segments


In [100]:
# Process dev data
print("Processing dev data...")
processed_dev = []

for i, doc in enumerate(tqdm(dev_documents, desc="Processing dev")):
    processed = align_labels_with_tokens(
        doc['text'],
        doc['entities'],
        tokenizer,
        label2id
    )
    processed['pmid'] = doc['pmid']
    processed['location'] = doc['location']
    processed['text'] = doc['text']
    processed['entities'] = doc['entities']
    processed_dev.append(processed)

print(f"✓ Dev data processed: {len(processed_dev)} segments")

Processing dev data...


Processing dev: 100%|██████████| 80/80 [00:00<00:00, 487.81it/s]

✓ Dev data processed: 80 segments


In [101]:
# Show example with BIO tags
example_idx = 10
example = processed_train[example_idx]

print(f"Example from training data:")
print(f"  Text: {example['text'][:150]}...")
print(f"  Entities: {len(example['entities'])}")
print(f"\nToken-Label pairs (first 30):")

token_label_pairs = []
for token, label_id in zip(example['tokens'][:30], example['labels'][:30]):
    label = id2label[label_id]
    token_label_pairs.append((token, label))

df = pd.DataFrame(token_label_pairs, columns=['Token', 'Label'])
print(df.to_string(index=False))

Example from training data:
  Text: A systematic review on gut-brain axis aberrations in bipolar disorder and methods of balancing the gut microbiota....
  Entities: 2

Token-Label pairs (first 30):
     Token        Label
     [CLS]            O
         A            O
systematic            O
    review            O
        on            O
       gut            O
         -            O
     brain            O
      axis            O
         a            O
     ##ber            O
 ##rations            O
        in            O
        bi        B-DDF
     ##pol        I-DDF
      ##ar        I-DDF
  disorder        I-DDF
       and            O
   methods            O
        of            O
 balancing            O
       the            O
       gut B-microbiome
     micro I-microbiome
     ##bio I-microbiome
      ##ta I-microbiome
         .            O
     [SEP]            O


## Prepare Dataset for BERT Training

In [102]:
class NERDataset(Dataset):
    """Custom dataset for NER token classification."""
    
    def __init__(self, processed_data, max_length=512):
        self.data = processed_data
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Pad or truncate to max_length
        input_ids = item['input_ids'][:self.max_length]
        attention_mask = item['attention_mask'][:self.max_length]
        labels = item['labels'][:self.max_length]
        
        # Pad if necessary
        padding_length = self.max_length - len(input_ids)
        if padding_length > 0:
            input_ids = input_ids + [tokenizer.pad_token_id] * padding_length
            attention_mask = attention_mask + [0] * padding_length
            labels = labels + [-100] * padding_length  # -100 is ignored by loss
        
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }


print("✓ Custom dataset class defined")

✓ Custom dataset class defined


In [103]:
# Create datasets
print("Creating training datasets...")

train_dataset = NERDataset(processed_train)
dev_dataset = NERDataset(processed_dev)

print(f"✓ Training dataset: {len(train_dataset)} examples")
print(f"✓ Dev dataset: {len(dev_dataset)} examples")

Creating training datasets...
✓ Training dataset: 1636 examples
✓ Dev dataset: 80 examples


## Configure Training Arguments

In [104]:
# Setup data collator for token classification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, return_tensors="pt")
print("✓ Data collator initialized")

✓ Data collator initialized


In [105]:
# Define training arguments
training_args = TrainingArguments(
    output_dir=output_model_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

print("✓ Training configuration ready")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Learning rate: {training_args.learning_rate}")

✓ Training configuration ready
  Batch size: 8
  Epochs: 3
  Learning rate: 2e-05


## Train BERT Model

Note: This cell might take several minutes to hours depending on dataset size and hardware.

**Additional configurations to test:**
- Change hyperparameters (*learning_rate*, *batch_size*, *num_train_epochs*, *weight_decay*)
- Try different pre-trained models (e.g., "allenai/scibert_scivocab_uncased", "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")
- Experiment with max_length for longer contexts

In [106]:
# Initialize Trainer
print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator
)

print("✓ Trainer initialized")
print(f"  Training samples: {len(train_dataset)}")
print(f"  Evaluation samples: {len(dev_dataset)}")

Initializing Trainer...
✓ Trainer initialized
  Training samples: 1636
  Evaluation samples: 80


In [107]:
# Start training
print("="*60)
print("Starting model training...")
print("="*60)

import time
training_start_time = time.time()

train_result = trainer.train()

training_duration = time.time() - training_start_time

print("\n" + "="*60)
print("✓ TRAINING COMPLETED!")
print("="*60)
print(f"Training time: {training_duration/60:.2f} minutes")

Starting model training...


C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.464094,0.378654
2,0.315945,0.315164
3,0.267547,0.291981


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]
C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.75it/s]
C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.92it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'be


✓ TRAINING COMPLETED!
Training time: 154.44 minutes


## Save Trained Model

In [108]:
# Save the trained model
print("Saving trained model...")

os.makedirs(output_model_dir, exist_ok=True)
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"✓ Model saved to: {output_model_dir}")

Saving trained model...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]

✓ Model saved to: models/bert_biomedbert_ner_aligned


## Load Model for Inference

In [109]:
# Load the trained model for inference
print("Loading trained model for inference...")

inference_model = AutoModelForTokenClassification.from_pretrained(output_model_dir)
inference_tokenizer = AutoTokenizer.from_pretrained(output_model_dir)
inference_model.eval()

if torch.cuda.is_available():
    inference_model = inference_model.cuda()

print(f"✓ Model loaded from: {output_model_dir}")

Loading trained model for inference...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 536.39it/s, Materializing param=classifier.weight]                                      


✓ Model loaded from: models/bert_biomedbert_ner_aligned


## Inference Function

predict_entities(model, tokenizer, text, id2label)
- Tokenizza con offset mapping.
- Fa argmax dei logits.
- Trasforma in label stringhe.
- Ricostruisce entità scorrendo token per token:
   -  se B-: chiude eventuale entità precedente e ne apre una nuova
   -  se I- compatibile: estende end_idx
altrimenti: chiude entità

In [110]:
def predict_entities(model, tokenizer, text, id2label):
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True,
        max_length=512
    )

    offset_mapping = encoding.pop("offset_mapping")[0].numpy()

    if torch.cuda.is_available():
        encoding = {k: v.cuda() for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**encoding)
        pred_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    predicted_labels = [id2label[p] for p in pred_ids]

    entities = []
    current = None

    for label, (start_char, end_char) in zip(predicted_labels, offset_mapping):
        # skip special tokens
        if start_char == 0 and end_char == 0:
            continue

        # ✅ cast to python int (evita np.int64)
        start_char = int(start_char)
        end_char = int(end_char)

        if label.startswith("B-"):
            if current:
                entities.append(current)

            ent_label = label[2:]
            current = {
                "start_idx": start_char,
                "end_idx": end_char - 1,   # ✅ inclusive
                "label": ent_label,
                "text_span": text[start_char:end_char],
            }

        elif label.startswith("I-") and current:
            ent_label = label[2:]
            if ent_label == current["label"]:
                current["end_idx"] = end_char - 1  # ✅ inclusive
                current["text_span"] = text[current["start_idx"]:end_char]

        else:
            if current:
                entities.append(current)
                current = None

    if current:
        entities.append(current)

    return entities



print("✓ Inference function defined")

✓ Inference function defined


## Predict on Dev Set

In [111]:
# Run inference on all dev data
print("Running inference on dev set...")

# Group predictions by document
predictions = {}

for doc in tqdm(dev_documents, desc="Predicting"):
    pmid = doc['pmid']
    location = doc['location']
    text = doc['text']
    
    # Predict entities
    predicted_entities = predict_entities(
        inference_model,
        inference_tokenizer,
        text,
        id2label
    )
    
    # Add location to each entity
    for entity in predicted_entities:
        entity['location'] = location
    
    # Initialize document if not exists
    if pmid not in predictions:
        predictions[pmid] = {'entities': []}
    
    # Add entities to document
    predictions[pmid]['entities'].extend(predicted_entities)

print(f"✓ Inference completed: {len(predictions)} documents")
total_entities = sum(len(p['entities']) for p in predictions.values())
print(f"  Total entities predicted: {total_entities}")

Running inference on dev set...


Predicting: 100%|██████████| 80/80 [00:09<00:00,  8.66it/s]

✓ Inference completed: 40 documents
  Total entities predicted: 1179


## Save Predictions

In [112]:
# Save predictions to file
output_path = "C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER_aligned.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"Predictions saved to {output_path}")

Predictions saved to C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions/bert_NER.json


## Evaluate Performance

In [113]:
# Load evaluation functions from evaluate.py concepts
def remove_duplicated_entities(predictions):
    """Remove duplicated entities from predictions."""
    removed_count = 0
    for pmid in list(predictions.keys()):
        seen = set()
        deduped = []
        for ent in predictions[pmid]["entities"]:
            #key = (ent["start_idx"], ent["end_idx"], ent["location"])
            key = (ent["start_idx"], ent["end_idx"], ent["location"], ent["label"])

            if key not in seen:
                seen.add(key)
                deduped.append(ent)
            else:
                removed_count += 1
        predictions[pmid]["entities"] = deduped
    
    if removed_count > 0:
        print(f"Removed {removed_count} duplicated entities from predictions")

def remove_overlapping_entities_eval(predictions):
    """Remove overlapping entities, keeping longest spans."""
    removed_count = 0

    for pmid in list(predictions.keys()):
        original_len = len(predictions[pmid]['entities'])
        
        groups = {'title': [], 'abstract': []}
        for ent in predictions[pmid]['entities']:
            loc = ent["location"]
            groups[loc].append(ent)

        keepers = set()
        for loc in groups:
            group = groups[loc]
            group = sorted(group, key=lambda e: e["start_idx"])

            clusters = []
            cluster = []
            current_end = None

            for ent in group:
                if not cluster:
                    cluster = [ent]
                    current_end = ent["end_idx"]
                else:
                    if ent["start_idx"] < current_end:
                        cluster.append(ent)
                        if ent["end_idx"] > current_end:
                            current_end = ent["end_idx"]
                    else:
                        clusters.append(cluster)
                        cluster = [ent]
                        current_end = ent["end_idx"]
            if cluster:
                clusters.append(cluster)

            for clust in clusters:
                longest = clust[0]
                max_len = longest["end_idx"] - longest["start_idx"]
                for ent in clust[1:]:
                    length = ent["end_idx"] - ent["start_idx"]
                    if length > max_len:
                        longest = ent
                        max_len = length
                keepers.add((longest["start_idx"],
                             longest["end_idx"],
                             longest["location"]))

        deduped = []
        for ent in predictions[pmid]['entities']:
            key = (ent["start_idx"], ent["end_idx"], ent["location"])
            if key in keepers:
                deduped.append(ent)
                keepers.remove(key)

        predictions[pmid]["entities"] = deduped
        removed_count += (original_len - len(deduped))

    if removed_count > 0:
        print(f"Removed {removed_count} overlapping entities")

print("✓ Evaluation helper functions defined")

✓ Evaluation helper functions defined


In [114]:
def evaluate_ner(predictions, ground_truth):
    """Evaluate NER predictions against ground truth."""
    # Remove duplicated and overlapping entities
    remove_duplicated_entities(predictions)
    remove_overlapping_entities_eval(predictions)
    
    LEGAL_ENTITY_LABELS = [
        "anatomical location", "animal", "bacteria", "biomedical technique",
        "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
        "human", "microbiome", "statistical technique"
    ]
    
    ground_truth_NER = dict()
    count_annotated_entities_per_label = {}
    
    for pmid, article in ground_truth.items():
        if pmid not in ground_truth_NER:
            ground_truth_NER[pmid] = []
        for entity in article['entities']:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            entry = (start_idx, end_idx, location, text_span, label)
            ground_truth_NER[pmid].append(entry)
            
            if label not in count_annotated_entities_per_label:
                count_annotated_entities_per_label[label] = 0
            count_annotated_entities_per_label[label] += 1

    count_predicted_entities_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}
    count_true_positives_per_label = {label: 0 for label in list(count_annotated_entities_per_label.keys())}

    for pmid in predictions.keys():
        entities = predictions[pmid]['entities']
        
        for entity in entities:
            start_idx = int(entity["start_idx"])
            end_idx = int(entity["end_idx"])
            location = str(entity["location"])
            text_span = str(entity["text_span"])
            label = str(entity["label"]) 
            
            if label not in LEGAL_ENTITY_LABELS:
                continue

            if label in count_predicted_entities_per_label:
                count_predicted_entities_per_label[label] += 1

            entry = (start_idx, end_idx, location, text_span, label)
            if pmid in ground_truth_NER and entry in ground_truth_NER[pmid]:
                count_true_positives_per_label[label] += 1

    count_annotated_entities = sum(count_annotated_entities_per_label.values())
    count_predicted_entities = sum(count_predicted_entities_per_label.values())
    count_true_positives = sum(count_true_positives_per_label.values())

    micro_precision = count_true_positives / (count_predicted_entities + 1e-10)
    micro_recall = count_true_positives / (count_annotated_entities + 1e-10)
    micro_f1 = 2 * ((micro_precision * micro_recall) / (micro_precision + micro_recall + 1e-10))

    precision, recall, f1 = 0, 0, 0
    n = len(count_annotated_entities_per_label)
    for label in count_annotated_entities_per_label.keys():
        current_precision = count_true_positives_per_label[label] / (count_predicted_entities_per_label[label] + 1e-10) 
        current_recall = count_true_positives_per_label[label] / (count_annotated_entities_per_label[label] + 1e-10) 
        
        precision += current_precision
        recall += current_recall
        f1 += 2 * ((current_precision * current_recall) / (current_precision + current_recall + 1e-10))
    
    precision = precision / n
    recall = recall / n
    f1 = f1 / n

    return precision, recall, f1, micro_precision, micro_recall, micro_f1


# Evaluate
precision, recall, f1, micro_precision, micro_recall, micro_f1 = evaluate_ner(predictions, dev_data)

print("="*60)
print("BERT NER BASELINE RESULTS")
print("="*60)
print("\nMacro-averaged Metrics:")
print(f"  Macro-Precision: {precision:.4f}")
print(f"  Macro-Recall:    {recall:.4f}")
print(f"  Macro-F1 Score:  {f1:.4f}")

print("\nMicro-averaged Metrics:")
print(f"  Micro-Precision: {micro_precision:.4f}")
print(f"  Micro-Recall:    {micro_recall:.4f}")
print(f"  Micro-F1 Score:  {micro_f1:.4f}")
print("="*60)

BERT NER BASELINE RESULTS

Macro-averaged Metrics:
  Macro-Precision: 0.6453
  Macro-Recall:    0.6931
  Macro-F1 Score:  0.6520

Micro-averaged Metrics:
  Micro-Precision: 0.7498
  Micro-Recall:    0.7914
  Micro-F1 Score:  0.7700


## Example Predictions

In [115]:
# Show example predictions
print("Example Predictions:\n")

sample_pmids = list(dev_data.keys())[:5]

for pmid in sample_pmids:
    article = dev_data[pmid]
    pred = predictions[pmid]
    
    print(f"Document PMID: {pmid}")
    print(f"Title: {article['metadata']['title'][:100]}...")
    print(f"\nGold entities: {len(article['entities'])}")
    print(f"Predicted entities: {len(pred['entities'])}")
    
    # Show first few predicted entities
    print("\nSample predictions:")
    for entity in pred['entities'][:5]:
        print(f"  - '{entity['text_span']}' [{entity['label']}] in {entity['location']}")
    
    # Calculate match statistics
    gold_set = set()
    for entity in article['entities']:
        gold_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    pred_set = set()
    for entity in pred['entities']:
        pred_set.add((
            entity['start_idx'],
            entity['end_idx'],
            entity['location'],
            entity['text_span'],
            entity['label']
        ))
    
    correct = len(gold_set & pred_set)
    missed = len(gold_set - pred_set)
    wrong = len(pred_set - gold_set)
    
    print(f"\n✓ Correct: {correct}")
    print(f"✗ Missed: {missed}")
    print(f"✗ Wrong: {wrong}")
    print("-" * 80)
    print()

Example Predictions:

Document PMID: 36532064
Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation....

Gold entities: 19
Predicted entities: 26

Sample predictions:
  - 'CNS autoimmune inflammation' [DDF] in title
  - 'In' [chemical] in abstract
  - 'neurological diseases' [DDF] in abstract
  - 'bacteria' [bacteria] in abstract
  - 'CNS autoimmunity' [DDF] in abstract

✓ Correct: 16
✗ Missed: 3
✗ Wrong: 10
--------------------------------------------------------------------------------

Document PMID: 37212075
Title: IgA-Biome Profiles Correlate with Clinical Parkinson's Disease Subtypes....

Gold entities: 21
Predicted entities: 28

Sample predictions:
  - 'Clinical Parkinson's Disease Subtype' [DDF] in title
  - 'Parkinson's disease' [DDF] in abstract
  - 'neurodegenerative disorder' [DDF] in abstract
  - 'gut microbiota' [microbiome] in abstract
  - 'secretory IgA' [chemical] in abstract

✓ Correct: 11
✗ Missed: 10
✗ Wrong: 17
----------------

## Analysis: Entity Distribution by Label

In [116]:
from collections import Counter

# Count entities by label in predictions
pred_label_counts = Counter()
for pmid, pred in predictions.items():
    for entity in pred['entities']:
        pred_label_counts[entity['label']] += 1

# Count entities by label in gold standard
gold_label_counts = Counter()
for pmid, article in dev_data.items():
    for entity in article['entities']:
        gold_label_counts[entity['label']] += 1

print("Entity Distribution by Label:")
print("="*60)
print(f"{'Label':<25} {'Gold':<10} {'Predicted':<10}")
print("-"*60)

all_labels = set(gold_label_counts.keys()) | set(pred_label_counts.keys())
for label in sorted(all_labels):
    print(f"{label:<25} {gold_label_counts[label]:<10} {pred_label_counts[label]:<10}")

print("-"*60)
print(f"{'TOTAL':<25} {sum(gold_label_counts.values()):<10} {sum(pred_label_counts.values()):<10}")

Entity Distribution by Label:
Label                     Gold       Predicted 
------------------------------------------------------------
DDF                       379        387       
anatomical location       76         91        
animal                    73         79        
bacteria                  54         75        
biomedical technique      36         46        
chemical                  131        149       
dietary supplement        27         46        
drug                      60         73        
food                      26         0         
gene                      39         6         
human                     86         88        
microbiome                127        137       
statistical technique     3          2         
------------------------------------------------------------
TOTAL                     1117       1179      
